In [0]:
%run ../_shared/00_project_config

In [0]:
%run ../_shared/01_spark_utils

In [0]:
from pyspark.sql import functions as F

In [0]:
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

SOURCE_SYSTEM = "football-data.org"

RAW_TEAMS_ROOT = (
    f"{FOOTBALL_DATA_ORG_RAW_ROOT}/teams"
)

BRONZE_TABLE = (
    f"{BRONZE_SCHEMA}."
    "football_data_org_teams"
)


RAW_TEAM_FILE_NAMES = {
    "PL": "pl_premier_league.json",
    "PD": "pd_la_liga.json",
    "BL1": "bl1_bundesliga.json",
    "SA": "sa_serie_a.json",
    "FL1": "fl1_ligue_1.json",
}

# -----------------------------------------------------------------------------
# Load football-data.org team JSON files into Bronze
# -----------------------------------------------------------------------------
#
# Each raw JSON file contains a response envelope with fields such as:
#
#   count
#   filters
#   competition
#   season
#   teams[]
#
# Bronze removes only the API response envelope by exploding teams[].
#
# Resulting Bronze grain:
#
#   one row = one football-data.org team within the competition snapshot
#
# Source field names and nested source structures are preserved.
#
# Technical metadata:
#
#   _league
#   _ingested_at
#   _source_file
#   _source_system
# -----------------------------------------------------------------------------

loaded_file_count = 0
loaded_team_count = 0


for competition_code, league in FOOTBALL_DATA_ORG_COMPETITIONS.items():

    file_name = RAW_TEAM_FILE_NAMES[
        competition_code
    ]

    file_path = (
        f"{RAW_TEAMS_ROOT}/"
        f"{file_name}"
    )
    source_hash = calculate_file_sha256(file_path)
    # -------------------------------------------------------------------------
    # Read raw JSON
    # -------------------------------------------------------------------------
    # Raw files are pretty-printed JSON documents, therefore multiLine
    # must be enabled.
    # -------------------------------------------------------------------------

    source_df = (
        spark.read
        .option("multiLine", "true")
        .json(file_path)
    )

    # -------------------------------------------------------------------------
    # Explode teams[]
    # -------------------------------------------------------------------------
    #
    # Keep competition and season context from the response envelope.
    # -------------------------------------------------------------------------

    bronze_df = (
        source_df
        .select(
            "competition",
            "season",
            F.explode("teams").alias("team"),
            F.col("_metadata.file_path").alias("_source_file"),
        )
        .select(
            "competition",
            "season",
            "team.*",
            "_source_file",
        )
        .withColumn(
            "_season_id",
            F.col("season.id").cast("long"),
        )
        .withColumn(
            "_league",
            F.lit(league),
        )
        .withColumn(
            "_ingested_at",
            F.current_timestamp(),
        )
        .withColumn(
            "_source_system",
            F.lit(SOURCE_SYSTEM),
        )
    )

    # -------------------------------------------------------------------------
    # Normalize nested coach.id schema
    # -------------------------------------------------------------------------
    # Spark infers coach.id as BIGINT from the incoming JSON.
    # The existing Bronze Delta table stores coach.id as STRING.
    #
    # Preserve the source-oriented coach structure while aligning
    # the identifier type with the existing Bronze schema.
    #
    # This avoids Delta schema merge conflicts during incremental
    # snapshot replacement.

    bronze_df = bronze_df.withColumn(
        "coach",
        F.col("coach").withField(
            "id",
            F.col("coach.id").cast("string"),
        ),
    )


    # -------------------------------------------------------------------------
    # Normalize staff to match the existing Bronze schema
    # -------------------------------------------------------------------------
    # Most API responses infer staff as array<string>, while some
    # contain staff objects (array<struct>).
    #
    # Serialize staff objects as JSON strings so all leagues use
    # array<string>, matching the existing Delta table.
    #
    # The complete staff object is preserved as JSON text.

    from pyspark.sql.types import ArrayType, StructType

    staff_type = bronze_df.schema["staff"].dataType

    if (
        isinstance(staff_type, ArrayType)
        and isinstance(staff_type.elementType, StructType)
    ):
        bronze_df = bronze_df.withColumn(
            "staff",
            F.transform(
                F.col("staff"),
                lambda staff_member: F.to_json(staff_member),
            ),
        )


    # -------------------------------------------------------------------------
    # Incremental loading: identify league-season snapshot
    # -------------------------------------------------------------------------

    season_ids = [
        row["_season_id"]
        for row in (
            bronze_df
            .select("_season_id")
            .distinct()
            .collect()
        )
    ]

    if len(season_ids) != 1 or season_ids[0] is None:
        raise ValueError(
            f"Expected one valid season in {file_path}. "
            f"Found: {season_ids}"
        )

    replace_predicate = (
        f"_league = '{league}' "
        f"AND _season_id = {season_ids[0]}"
    )

    if not bronze_file_needs_processing(
        spark=spark,
        table_name=BRONZE_TABLE,
        replace_predicate=replace_predicate,
        source_hash=source_hash,
    ):
        print(
            f"SKIP unchanged API teams: {file_path}"
        )
        continue

    team_count = bronze_df.count()

    # -------------------------------------------------------------------------
    # Write to Bronze Delta table
    # -------------------------------------------------------------------------

    (

    write_bronze_file_snapshot(
        df=bronze_df,
        table_name=BRONZE_TABLE,
        replace_predicate=replace_predicate,
        source_hash=source_hash,
    )

    )

    loaded_file_count += 1
    loaded_team_count += team_count

    print(
        f"Loaded football-data.org Bronze teams file "
        f"{loaded_file_count}: "
        f"{competition_code} / {league} | "
        f"teams={team_count} "
        f"-> {BRONZE_TABLE}"
    )

# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {loaded_file_count} "
    "football-data.org team source files into Bronze. "
    f"Total teams loaded: {loaded_team_count}."
)